# atalla-sim — CuPy GPU Simulation

Cycle-accurate SoC simulator with CuPy-accelerated systolic array compute.

> **Before running:** Runtime → Change runtime type → **T4 GPU**

## 1. Environment setup

In [ ]:
# Confirm a GPU is present
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
# Clone repo
!git clone https://github.com/Purdue-SoCET/atalla-sim.git
%cd atalla-sim

In [ ]:
# Install CuPy matching the Colab CUDA version (12.x on most Colab runtimes)
import subprocess
cuda_out = subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout
pkg = 'cupy-cuda12x' if 'release 12' in cuda_out else 'cupy-cuda11x' if 'release 11' in cuda_out else 'cupy'
print(f'Installing {pkg}')
!pip install -q {pkg}

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import cupy as cp
from systolic_array.systolic_array_tpu import _USE_CUPY

gpu = cp.cuda.Device(0)
print(f'CuPy     : {cp.__version__}')
print(f'GPU      : {gpu.attributes["name"] if hasattr(gpu, "attributes") else "T4"}')
print(f'VRAM     : {cp.cuda.runtime.memGetInfo()[1] / 1e9:.1f} GB')
print(f'_USE_CUPY: {_USE_CUPY}')
assert _USE_CUPY, 'GPU kernel not active — switch runtime to T4 GPU and re-run'

## 2. Micro-benchmark: GPU vs pure-Python tick()

In [ ]:
import time, random
import systolic_array.systolic_array_tpu as _m
from systolic_array.systolic_array_tpu import SystolicArrayTPU

SIZE, GS, N = 32, 4, 300

def make_sa(gpu: bool):
    # _xp is fixed at import time; patch both flags so __init__ allocates correctly
    import numpy as np
    _m._USE_CUPY = gpu
    _m._xp = cp if gpu else np
    weights = [[random.uniform(-1, 1) for _ in range(SIZE)] for _ in range(SIZE)]
    sa = SystolicArrayTPU(size=SIZE, dtype='fp16', group_size=GS)
    sa.load_weights(weights)
    return sa

def drive(sa, n):
    for _ in range(n + SIZE + 5):
        vec = [random.uniform(-1, 1) for _ in range(sa.group_size)]
        for g in range(sa.num_groups):
            sa._input_fifo_left[g].enqueue(vec)
        sa._input_algo_flags.enqueue(True)
    sa.mac_shift = True; sa.start = True
    for t in range(n):
        sa.tick(t)

def bench(gpu: bool, label: str):
    sa = make_sa(gpu); drive(sa, 20)          # warmup
    sa = make_sa(gpu)
    t0 = time.perf_counter(); drive(sa, N); t1 = time.perf_counter()
    us = (t1 - t0) / N * 1e6
    print(f'{label:12s}: {us:7.1f} µs/tick')
    return us

us_gpu = bench(True,  'GPU (CuPy)')
us_cpu = bench(False, 'CPU (Python)')
print(f'\nSpeedup    : {us_cpu / us_gpu:.1f}x')

# Restore GPU mode for subsequent cells
_m._USE_CUPY = True
_m._xp = cp

## 3. Full end-to-end simulation (32×32 tile, fp16)

In [ ]:
import time
from atalla.sysarr_tpu_experiment import SysArrTPUExperimentConfig, run_sysarr_tpu_experiment

cfg = SysArrTPUExperimentConfig(
    name='colab_gpu_demo',
    sweep='demo',
    param_name='tile',
    param_value='32',
    tile=32,
    dtype='fp16',
    lane_count=4,
    spad_num_banks=32,
    spad_bank_size=128,
    max_cycles=50000,
)

t0 = time.perf_counter()
result = run_sysarr_tpu_experiment(cfg)
elapsed = time.perf_counter() - t0

print(f'Simulated cycles    : {result["cycles"]}')
print(f'MAC utilization     : {result["mac_utilization"]:.1%}')
print(f'Throughput          : {result["throughput_flops_per_cycle"]:.2f} FLOP/cycle')
print(f'External BW         : {result["external_bw_bytes_per_cycle"]:.2f} B/cycle')
print(f'Wall-clock sim time : {elapsed:.1f} s')

## 4. Correctness check

In [ ]:
!python -m pytest tests/ \
    --ignore=tests/atalla/test_scratchpad_vector_core_sysarr_tpu_tiled_1024.py \
    --ignore=tests/atalla/test_scratchpad_vector_core_sysarr_tpu_tiled_1024_blocked_mn.py \
    -q